## Investigate which files are missing after assignment for each Replicate
- load assignment file (/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/experiments/standard_bwa/assignment/assignmentFixDuplicates.tsv.gz)
- load barcode files for each replicate
  - merge files based on barcode and count unique sequences

In [45]:
import pandas as pd 
import yaml 


config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [46]:
assignment_file = pd.read_csv(config['files']['final_design']['assignment_file'], header=None, sep='\t')
assignment_file.columns = ['barcode', 'header', 'alignment_info', 'matching_number']


In [49]:
assignment_file.head()

,barcode,header,alignment_info,matching_number
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,15;270M;NM:i:0;MD:Z:270;60,5/5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,6/7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,15;270M;NM:i:0;MD:Z:270;6,9/10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,15;270M;NM:i:0;MD:Z:270;6,7/7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,7/8


In [47]:
barcode_rep1 = pd.read_csv(config['files']['final_design']['barcodes_rep1'], header=None, sep='\t')
barcode_rep1.columns = ['barcode', 'dna_count', 'rna_count']
barcode_rep1

,barcode,dna_count,rna_count
0,AAAAAAAAAACAAGT,1,1
1,AAAAAAAAAAGCTGG,4,11
2,AAAAAAAAAATCCTA,2,6
3,AAAAAAAAAATCTAC,3,12
4,AAAAAAAAAATGTTT,4,4
...,...,...,...
4307921,TTTTTTTTTGGGGGA,1,3
4307922,TTTTTTTTTGTTAAA,3,2
4307923,TTTTTTTTTTCTGGC,2,6
4307924,TTTTTTTTTTGACCG,5,11


In [48]:
barcode_rep2 = pd.read_csv(config['files']['final_design']['barcodes_rep2'], header=None, sep='\t')
barcode_rep2.columns = ['barcode', 'dna_count', 'rna_count']
barcode_rep3 = pd.read_csv(config['files']['final_design']['barcodes_rep3'], header=None, sep='\t')
barcode_rep3.columns = ['barcode', 'dna_count', 'rna_count']

In [50]:
# merge barcode file per replicate to the assignment file and check unique number of sequences 
assignment_barcode_rep1 = barcode_rep1.merge(assignment_file, on='barcode', how='left')
assignment_barcode_rep2 = barcode_rep2.merge(assignment_file, on='barcode', how='left')
assignment_barcode_rep3 = barcode_rep3.merge(assignment_file, on='barcode', how='left')

In [52]:
assignment_barcode_rep1.head()

,barcode,dna_count,rna_count,header,alignment_info,matching_number
0,AAAAAAAAAACAAGT,1,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,6/7
1,AAAAAAAAAAGCTGG,4,11,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,7/8
2,AAAAAAAAAATCCTA,2,6,cardiac_neuro_cava_random:ALT_ANK3|ENSG0000015...,15;270M;NM:i:0;MD:Z:270;6,15/16
3,AAAAAAAAAATCTAC,3,12,cardiac_neuro_cava_random:ALT_NBEA|ENSG0000017...,15;270M;NM:i:0;MD:Z:270;6,19/20
4,AAAAAAAAAATGTTT,4,4,cardiac_neuro_cava_random:ALT_CFAP91|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,8/10


In [55]:
rep1_header_set = set(assignment_barcode_rep1['header'])
rep2_header_set = set(assignment_barcode_rep2['header'])
rep3_header_set = set(assignment_barcode_rep3['header'])


In [58]:
# print number of unique sequences in each replicate
print('rep1:', assignment_barcode_rep1['header'].nunique())
print('rep2:', assignment_barcode_rep2['header'].nunique())
print('rep3:', assignment_barcode_rep3['header'].nunique())

rep1: 74689
rep2: 74661
rep3: 74660


In [57]:
# get all headers that are in all replicates
common_headers = rep1_header_set.intersection(rep2_header_set).intersection(rep3_header_set)
print('Number of sequences in all headers: ', len(common_headers)) # 74401

Number of sequences in all headers:  74401


In [54]:
# control not control sequences (control: if label != cardiac_neuro_cava_random) 
# get the label of the header
assignment_barcode_rep1['label'] = assignment_barcode_rep1['header'].apply(lambda x: 'control' if 'control' in x else 'not_control')

rep1: 74689
rep2: 74661
rep3: 74660


In [83]:
print("Number of missing rep1: ", 80215 - assignment_barcode_rep1['header'].nunique())
print("Number of missing rep2: ", 80215 - assignment_barcode_rep2['header'].nunique())
print("Number of missing rep3: ", 80215 - assignment_barcode_rep3['header'].nunique())

Number of missing rep1:  5526
Number of missing rep2:  5554
Number of missing rep3:  5555


#### Investigate common_headers (control / tested + ref/alt)


In [80]:
common_sequences = designed_sequences[designed_sequences['header'].isin(common_headers)]
common_sequences['label'] = common_sequences['header'].str.split(':').str[0]
common_sequences['is_control'] = common_sequences['label'].apply(lambda x: True if x != 'cardiac_neuro_cava_random' else False)
common_sequences['is_control'].value_counts()

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/3509603581.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  common_sequences['label'] = common_sequences['header'].str.split(':').str[0]
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/3509603581.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  common_sequences['is_control'] = common_sequences['label'].apply(lambda x: True if x != 'cardiac_neuro_cava_random' else False)


is_control
False    68773
True      5557
Name: count, dtype: int64

In [81]:
# investigate if ALT REF or Region
tested_sequences_found = common_sequences[common_sequences['is_control'] == False]
tested_sequences_found['is_REF'] = tested_sequences_found['header'].apply(lambda x: True if ':REF_' in x else False)
tested_sequences_found['is_ALT'] = tested_sequences_found['header'].apply(lambda x: True if ':ALT_' in x else False)
tested_sequences_found['is_Region'] = tested_sequences_found.apply(lambda x: True if not (x['is_REF'] or x['is_ALT']) else False, axis=1)
print('Number of regions in found tested sequences: ', tested_sequences_found[tested_sequences_found['is_Region'] == True].shape[0]) # 8544
print('Number of REF in found tested sequences: ', tested_sequences_found[tested_sequences_found['is_REF'] == True].shape[0]) # 17327
print('Number of ALT in found tested sequences: ', tested_sequences_found[tested_sequences_found['is_ALT'] == True].shape[0]) # 42902

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/4179186179.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_sequences_found['is_REF'] = tested_sequences_found['header'].apply(lambda x: True if ':REF_' in x else False)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/4179186179.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_sequences_found['is_ALT'] = tested_sequences_found['header'].apply(lambda x: True if ':ALT_' in x else False)


Number of regions in found tested sequences:  8544
Number of REF in found tested sequences:  17327
Number of ALT in found tested sequences:  42902


/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/4179186179.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_sequences_found['is_Region'] = tested_sequences_found.apply(lambda x: True if not (x['is_REF'] or x['is_ALT']) else False, axis=1)


#### Investigate missing sequences

In [60]:
# get all headers which are not in common_headers 
designed_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])
designed_sequences

designed_header = set(designed_sequences['header'])
missing_headers = designed_header - common_headers
print('Number of missing headers after MPRAsnakeflow:', len(missing_headers)) # 5885

5885

In [73]:
# get label of the missing headers (header.split(":")[0]) and classify: control: label != "cardiac_neuro_cava_random"
missing_sequences = designed_sequences[designed_sequences['header'].isin(missing_headers)]
missing_sequences['label'] = missing_sequences['header'].str.split(':').str[0]
missing_sequences['is_control'] = missing_sequences['label'].apply(lambda x: True if x != 'cardiac_neuro_cava_random' else False)
missing_sequences['is_control'].value_counts()
# False    5167
# True      718



Number of regions in missing tested sequences:  356
Number of REF in missing tested sequences:  1255
Number of ALT in missing tested sequences:  3556


/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/2235940413.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_sequences['label'] = missing_sequences['header'].str.split(':').str[0]
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/2235940413.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_sequences['is_control'] = missing_sequences['label'].apply(lambda x: True if x != 'cardiac_neuro_cava_random' else False)
/data/gpfs-1/users/kisa11_c/scratch/tmp/h

#### Investigate distribution of missing control sequences:
- which control group has how many missing sequences?

In [78]:
missing_controls = missing_sequences[missing_sequences['is_control'] == True]
pd.DataFrame(missing_controls['label'].value_counts()).reset_index()

# label
# MK                                   220
# C_positive_heart_AB                  190
# C_negative_neuron_NP                  54
# GC_DNase_positive                     41
# C_SLEA                                26
# GC_Kircher                            23
# C_negative_neuron_MK                  22
# GC_Vista                              21
# GC_DNase_negative_brain               15
# GC_DNase_negative_blood               15
# GC_GABA_Chengyu                       14
# C_negative_heart_MK                   13
# GC_Selvarajan                         12
# C_positive_heart_MK                   11
# C_positive_neuron_CD                   9
# GC_Glut_Chengyu                        8
# C_positive_heart_CAD                   7
# C_positive_neuron_NP                   6
# GC_Mendelian_variants                  5
# GC_Mohlke                              3
# GC_DNase_positive_shuffeled            2
# GC_DNase_negative_blood_shuffeled      1

,label,count
0,MK,220
1,C_positive_heart_AB,190
2,C_negative_neuron_NP,54
3,GC_DNase_positive,41
4,C_SLEA,26
5,GC_Kircher,23
6,C_negative_neuron_MK,22
7,GC_Vista,21
8,GC_DNase_negative_brain,15
9,GC_DNase_negative_blood,15


#### Investigate missing tested sequences:
- How many regions, refs and variants are we losing?

In [82]:
tested_sequences_missing = missing_sequences[missing_sequences['is_control'] == False]
tested_sequences_missing['is_REF'] = tested_sequences_missing['header'].apply(lambda x: True if ':REF_' in x else False)
tested_sequences_missing['is_ALT'] = tested_sequences_missing['header'].apply(lambda x: True if ':ALT_' in x else False)
tested_sequences_missing['is_Region'] = tested_sequences_missing.apply(lambda x: True if not (x['is_REF'] or x['is_ALT']) else False, axis=1)
print('Number of regions in missing tested sequences: ', tested_sequences_missing[tested_sequences_missing['is_Region'] == True].shape[0]) # 356
print('Number of REF in missing tested sequences: ', tested_sequences_missing[tested_sequences_missing['is_REF'] == True].shape[0]) # 1255
print('Number of ALT in missing tested sequences: ', tested_sequences_missing[tested_sequences_missing['is_ALT'] == True].shape[0]) # 3556

Number of regions in missing tested sequences:  356
Number of REF in missing tested sequences:  1255
Number of ALT in missing tested sequences:  3556


/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/1376186970.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_sequences_missing['is_REF'] = tested_sequences_missing['header'].apply(lambda x: True if ':REF_' in x else False)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/1376186970.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_sequences_missing['is_ALT'] = tested_sequences_missing['header'].apply(lambda x: True if ':ALT_' in x else False)
/data/gp

In [71]:
tested_sequences_missing[tested_sequences_missing['is_Region'] == True]

,header,sequence,label,is_control,is_REF,is_ALT,is_Region
12,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACCTGTGTCCCGCCAGCAGTAGGGCCTAGCA...,cardiac_neuro_cava_random,False,False,False,True
24,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTTTATTAGAGACAGGGTCTCACTCTGTCGC...,cardiac_neuro_cava_random,False,False,False,True
29,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGAAGAGACCTTGGGTCTCAGTTGTGCC...,cardiac_neuro_cava_random,False,False,False,True
33,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCTGGCCACCCCCACGTCCCAGCACCTGCTTT...,cardiac_neuro_cava_random,False,False,False,True
35,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGCCTGGAGTGGGTAGTGCCAGGCAGAGGC...,cardiac_neuro_cava_random,False,False,False,True
...,...,...,...,...,...,...,...
8870,cardiac_neuro_cava_random:MECP2|ENSG0000016905...,AGGACCGGATCAACTTTTCAGCACTGAGCTAGCCTCTCCTTGCTAG...,cardiac_neuro_cava_random,False,False,False,True
8871,cardiac_neuro_cava_random:MECP2|ENSG0000016905...,AGGACCGGATCAACTAACTGGGGCGGGGTGGATCACTCCAAACTTG...,cardiac_neuro_cava_random,False,False,False,True
8889,cardiac_neuro_cava_random:MECP2|ENSG0000016905...,AGGACCGGATCAACTTAAGTGTATAGTCCAGTCACCACCATCCGTC...,cardiac_neuro_cava_random,False,False,False,True
8893,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTGTCAGCGTGAAGTACAAGGGCCAGCACGTGC...,cardiac_neuro_cava_random,False,False,False,True
